# Problem 2: Regularization and Dropout on Fashion MNIST

Comparing two MLP models trained for 40 epochs:
- **Model 1**: 1 hidden layer (128 nodes, ReLU), no regularization, no dropout
- **Model 2**: 1 hidden layer (48 nodes, ReLU), L2 regularization (λ=0.0001), dropout (p=0.2)

We examine how these design choices affect the learned weight distributions.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    device = torch.device('cuda')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Using device: {device}')

In [ ]:
# Load Fashion MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_data = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
test_data = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_data, batch_size=100, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=100, shuffle=False)

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
print(f'Train: {len(train_data)} samples, Test: {len(test_data)} samples')

In [ ]:
class Model1(nn.Module):
    """1 hidden layer, 128 nodes, ReLU - no regularization, no dropout."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


class Model2(nn.Module):
    """1 hidden layer, 48 nodes, ReLU + L2 reg (weight_decay) + dropout(0.2)."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 48)
        self.fc2 = nn.Linear(48, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=0.2)
    
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.dropout(self.relu(self.fc1(x)))
        return self.fc2(x)

print('Model 1:')
print(Model1())
print('\nModel 2:')
print(Model2())

In [ ]:
def train_model(model, train_loader, test_loader, optimizer, n_epochs=40):
    criterion = nn.CrossEntropyLoss()
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    
    for epoch in range(n_epochs):
        # training phase
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(yb)
            correct += (logits.argmax(1) == yb).sum().item()
            total += len(yb)
        train_losses.append(total_loss / total)
        train_accs.append(correct / total)
        
        # evaluation phase
        model.eval()
        total_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for xb, yb in test_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = criterion(logits, yb)
                total_loss += loss.item() * len(yb)
                correct += (logits.argmax(1) == yb).sum().item()
                total += len(yb)
        test_losses.append(total_loss / total)
        test_accs.append(correct / total)
        
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | '
                  f'train loss: {train_losses[-1]:.4f} | '
                  f'test acc: {test_accs[-1]:.4f}')
    
    return train_losses, test_losses, train_accs, test_accs

In [ ]:
# Train Model 1 - no regularization
model1 = Model1().to(device)
opt1 = optim.Adam(model1.parameters(), lr=0.001)

print('Training Model 1 (128 nodes, no regularization)...')
tr1, te1, tra1, tea1 = train_model(model1, train_loader, test_loader, opt1, n_epochs=40)
print(f'Model 1 final test accuracy: {tea1[-1]*100:.2f}%')

In [ ]:
# Train Model 2 - L2 regularization + dropout
model2 = Model2().to(device)
opt2 = optim.Adam(model2.parameters(), lr=0.001, weight_decay=1e-4)

print('Training Model 2 (48 nodes, L2 + dropout)...')
tr2, te2, tra2, tea2 = train_model(model2, train_loader, test_loader, opt2, n_epochs=40)
print(f'Model 2 final test accuracy: {tea2[-1]*100:.2f}%')

In [ ]:
# Weight histograms for Model 1
w1_input  = model1.fc1.weight.detach().cpu().numpy().flatten()
w1_hidden = model1.fc2.weight.detach().cpu().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Model 1 Weight Distributions (128 nodes, no regularization)', fontsize=13)

axes[0].hist(w1_input, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Input Layer (fc1) Weights')
axes[0].set_xlabel('Weight Value')
axes[0].set_ylabel('Count')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)

axes[1].hist(w1_hidden, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Hidden Layer (fc2) Weights')
axes[1].set_xlabel('Weight Value')
axes[1].set_ylabel('Count')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig('model1_weights.pdf', bbox_inches='tight')
plt.show()
print('Saved: model1_weights.pdf')
print(f'Model 1 fc1 weight std: {w1_input.std():.4f}')
print(f'Model 1 fc2 weight std: {w1_hidden.std():.4f}')

In [ ]:
# Weight histograms for Model 2
w2_input  = model2.fc1.weight.detach().cpu().numpy().flatten()
w2_hidden = model2.fc2.weight.detach().cpu().numpy().flatten()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Model 2 Weight Distributions (48 nodes, L2 + dropout)', fontsize=13)

axes[0].hist(w2_input, bins=60, color='darkorange', edgecolor='white', alpha=0.8)
axes[0].set_title('Input Layer (fc1) Weights')
axes[0].set_xlabel('Weight Value')
axes[0].set_ylabel('Count')
axes[0].axvline(0, color='red', linestyle='--', linewidth=1)

axes[1].hist(w2_hidden, bins=60, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].set_title('Hidden Layer (fc2) Weights')
axes[1].set_xlabel('Weight Value')
axes[1].set_ylabel('Count')
axes[1].axvline(0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.savefig('model2_weights.pdf', bbox_inches='tight')
plt.show()
print('Saved: model2_weights.pdf')
print(f'Model 2 fc1 weight std: {w2_input.std():.4f}')
print(f'Model 2 fc2 weight std: {w2_hidden.std():.4f}')

## Analysis: Effect of Regularization on Weight Distributions

### Model 1 (128 nodes, no regularization, no dropout)

The weights in Model 1 show a relatively broad distribution centered near zero. Without any regularization, the optimizer is free to assign whatever magnitude is needed to minimize the training loss. The input layer weights (fc1) have a noticeably wider spread compared to the hidden-to-output layer (fc2), which makes sense given that fc1 has to learn useful representations from 784 raw pixel values. There are weights with fairly large absolute values, reflecting the model's ability to "focus" strongly on specific features.

### Model 2 (48 nodes, L2 regularization λ=0.0001, dropout p=0.2)

The weight distributions in Model 2 are significantly more concentrated around zero. L2 regularization adds a penalty proportional to the squared weight values to the loss function, which pushes the optimizer to keep weights small unless they provide a strong benefit to classification accuracy. The result is a tighter, more Gaussian-shaped distribution with less variance.

Dropout also plays an indirect role: since any hidden unit can be zeroed out randomly during training, the network cannot rely on individual neurons and must spread responsibility across multiple units. This tends to produce more uniform weight magnitudes across the layer.

### Key Qualitative Differences

- **Spread**: Model 2 weights have smaller standard deviation, indicating more compact distributions.
- **Tails**: Model 1 has heavier tails — some weights take on large positive or negative values. Model 2 has lighter tails due to L2 penalty discouraging extreme values.
- **Shape**: Both models show roughly bell-shaped (approximately Gaussian) distributions, but Model 2's distribution is more sharply peaked at zero, which is the expected behavior of L2 regularization.
- **Interpretation**: The regularized model distributes information more evenly across its (fewer) weights, whereas the unregularized model can afford to ignore many weights while amplifying a few important ones.